In [27]:
import minsearch
import pandas as pd

In [28]:
file_path = "youtube_transcripts/dataset.json"

In [29]:
df = pd.read_json(file_path)

In [9]:
df.columns

Index(['question', 'answer', 'video_id'], dtype='str')

In [30]:
documents = df.to_dict(orient='records')

In [31]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
openai_client = OpenAI()

In [32]:
def search(question):
    
    index = minsearch.Index(
    text_fields=['question', 'answer', 'video_id'],
    keyword_fields=[]
)
    index.fit(documents)
    return index.search(
        question,
        num_results=5
    )

In [36]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

In [17]:
USER_PROMPT_TEMPLATE = '''

You are a helpful assistant that answers questions regarding self-awareness based on the provided context.

Use the context to find relevant information and provide accurate
answers. 
With the answer, provide one link to the YouTube video where the answer can be found.
If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''.strip()

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('Suggested Video: https://www.youtube.com/watch?v=' + doc['video_id'])
        lines.append(' ')

    return '\n'.join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [9]:
query="What is emotional intelligence?"

In [33]:
search_results = search(query)
prompt = build_prompt(query, search_results)

In [34]:
print(prompt)

You are a helpful assistant that answers questions regarding self-awareness based on the provided context.

Use the context to find relevant information and provide accurate
answers. 
With the answer, provide one link to the YouTube video where the answer can be found.
If the answer is not found in the context,
respond with "I don't know."

Question:
What is emotional intelligence?

Context:
Q: What is emotional intelligence?
A: Emotional intelligence is how effectively you handle yourself, your emotions, and your relationships with other people.
URL https://www.youtube.com/watch?v=BqF50IuR3_c
 
Q: What is emotional self-management?
A: Emotional self-management is the ability to handle distressing emotions effectively so they do not interfere with what you are doing.
URL https://www.youtube.com/watch?v=BqF50IuR3_c
 
Q: How is emotional intelligence connected to everyday life?
A: Emotional intelligence affects how effectively you manage yourself and interact with other people.
URL https

In [37]:
answer = llm(prompt)

In [38]:
print(answer)

Emotional intelligence is how effectively you handle yourself, your emotions, and your relationships with other people.  
https://www.youtube.com/watch?v=BqF50IuR3_c


In [39]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm( prompt)
    return answer

In [40]:
print(rag("Which techniques can help improve self-awareness?"))

Techniques that can help improve self-awareness include:

- Watching a recording of yourself to notice gestures, posture, tone, and speaking habits you may not realize you have
- Asking yourself what causes your stress, who triggers it, and when it happens
- Reflecting on your emotions by asking why you feel afraid, guilty, sad, angry, or irritated, and what caused those feelings

YouTube link: https://www.youtube.com/watch?v=ln8rIBZbWAE
